# Scrape the first three eCatholic parish directories
Run all cells from the project root. Requires `pandas`, `requests`, and `beautifulsoup4`.
The first three matching dioceses are selected in CSV order. All directory entries are retained (directories can also include missions, chapels, and other institutions); `category` preserves their source grouping.
The combined `parishes_df` is exported to `1_create_parish_list/ecatholic_parishes_first_3.csv` only after all three sites succeed.
Downloaded HTML is cached in `1_create_parish_list/html_cache/` and reused on later runs. Delete a cached HTML file to download that page again.


In [11]:
from pathlib import Path
from hashlib import sha256
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

input_path = Path("1_create_parish_list/diocese_site_info.csv")
output_path = input_path.with_name("ecatholic_parishes_first_3.csv")
cache_dir = input_path.parent / "html_cache"

dioceses_df = pd.read_csv(input_path)
ecatholic = dioceses_df["ecatholic?"].astype("string").str.strip().str.lower().eq("true")
selected_dioceses = dioceses_df.loc[ecatholic].copy()
if selected_dioceses["parish_list_url"].isna().any():
    raise ValueError("Expected eCatholic dioceses with parish directory URLs.")
selected_dioceses[["diocese_name", "parish_list_url"]]


,diocese_name,parish_list_url
0,Archdiocese of Mobile,https://mobarch.org/parishfinder
1,Diocese of Birmingham,https://www.bhmdiocese.org/parishfinder
2,Archdiocese of Anchorage-Juneau,https://www.aoaj.org/parishfinder
3,Diocese of Fairbanks,https://dioceseoffairbanks.org/parishfinder
6,Diocese of Tucson,https://diocesetucson.org/parishfinder
12,Diocese of Fresno,https://dioceseoffresno.org/parishfinder
13,Diocese of Monterey,https://www.dioceseofmonterey.org/parishfinder
20,Diocese of Santa Rosa,https://srdiocese.org/parishfinder
33,Diocese of Pensacola-Tallahassee,https://www.ptdiocese.org/parishfinder
49,Archdiocese of Indianapolis,https://www.archindy.org/parishfinder


In [ ]:
def get_page(url: str) -> tuple[bytes, str]:
    """Read cached HTML, or download and cache a successful response."""
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_key = sha256(url.encode("utf-8")).hexdigest()
    html_path = cache_dir / f"{cache_key}.html"
    url_path = cache_dir / f"{cache_key}.url"

    if html_path.is_file():
        # Preserve the final URL after redirects for relative website links.
        page_url = url_path.read_text(encoding="utf-8") if url_path.is_file() else url
        return html_path.read_bytes(), page_url

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    url_path.write_text(response.url, encoding="utf-8")
    # Rename only after the complete HTML has been written.
    temporary_path = html_path.with_suffix(".html.tmp")
    temporary_path.write_bytes(response.content)
    temporary_path.replace(html_path)
    return response.content, response.url


def scrape_parishes(diocese: str, url: str) -> pd.DataFrame:
    html, page_url = get_page(url)
    soup = BeautifulSoup(html, "html.parser")
    parishes = []

    for parish in soup.select("li.site .siteInfo"):
        name_element = parish.select_one(".title .name")
        if name_element is None or not name_element.get_text(strip=True):
            raise ValueError(f"Missing directory entry name for {diocese}: {url}")
        address_element = parish.select_one(".title .address")
        website = parish.select_one(".website a[href]")
        category = parish.find_parent("li", class_="category")
        category_name = category.select_one(".categoryName") if category else None
        parishes.append({
            "name": name_element.get_text(" ", strip=True),
            "address": address_element.get_text(" ", strip=True) if address_element else "",
            "site_url": urljoin(page_url, website["href"]) if website else "",
            "diocese": diocese,
            "category": category_name.get_text(" ", strip=True) if category_name else "",
            "parish_list_url": url,
        })

    if not parishes:
        raise ValueError(f"No parish directory entries found for {diocese}: {url}")
    return pd.DataFrame(parishes).drop_duplicates().reset_index(drop=True)


In [14]:
parish_frames = []
for row in selected_dioceses.itertuples(index=False):
    parish_df = scrape_parishes(str(row.diocese_name), str(row.parish_list_url))
    parish_frames.append(parish_df)
    print(f"{row.diocese_name}: {len(parish_df)} entries")

parishes_df = pd.concat(parish_frames, ignore_index=True).drop_duplicates().reset_index(drop=True)
parishes_df.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved {len(parishes_df)} entries to {output_path}")
parishes_df.head()


Archdiocese of Mobile: 86 entries
Diocese of Birmingham: 80 entries
Archdiocese of Anchorage-Juneau: 53 entries
Diocese of Fairbanks: 47 entries
Diocese of Tucson: 85 entries
Diocese of Fresno: 136 entries
Diocese of Monterey: 47 entries
Diocese of Santa Rosa: 61 entries
Diocese of Pensacola-Tallahassee: 57 entries
Archdiocese of Indianapolis: 132 entries
Diocese of Lafayette in Indiana: 60 entries
Archdiocese of Dubuque: 208 entries
Diocese of Davenport: 73 entries
Diocese of Sioux City: 121 entries
Archdiocese of New Orleans: 127 entries
Diocese of Baton Rouge: 81 entries
Diocese of Houma-Thibodaux: 42 entries
Diocese of Lafayette in Louisiana: 151 entries
Diocese of Springfield, Massachusetts: 81 entries
Diocese of Worcester: 103 entries
Archdiocese of Santa Fe: 94 entries
Diocese of Bismarck: 100 entries
Diocese of Fargo: 137 entries
Eparchy of St. George in Canton for the Romanians: 22 entries
Diocese of Steubenville: 54 entries
Archdiocese of Oklahoma City: 106 entries
Diocese of

,name,address,site_url,diocese,category,parish_list_url
0,Father Purcell Memorial Center for Exceptional...,"Montgomery, AL",https://fatherpurcell.org/,Archdiocese of Mobile,Sunday Mass Schedule,https://mobarch.org/parishfinder
1,Holy Spirit Catholic Parish,"Montgomery, AL",https://holyspiritmgm.org/,Archdiocese of Mobile,Sunday Mass Schedule,https://mobarch.org/parishfinder
2,Blessed Francis Xavier Seelos Parish,"31122 US Hwy 31, Spanish Fort, AL, 36527",https://francisxseelos.org/,Archdiocese of Mobile,Baldwin/Escambia Deanery,https://mobarch.org/parishfinder
3,Chapel of Our Lady of Bon Secour,"17266 County Road 49 South, Bon Secour, AL, 36511",https://www.stjohnms.com/our-lady-of-bon-secour,Archdiocese of Mobile,Baldwin/Escambia Deanery,https://mobarch.org/parishfinder
4,Christ the King Catholic Church,"711 College Avenue, Daphne, AL, 36526",https://ctkdaphne.org/,Archdiocese of Mobile,Baldwin/Escambia Deanery,https://mobarch.org/parishfinder
